# CXR Intelligence System — Full Pipeline (Single Notebook)

Runs everything in one session: data loading → indexing → demo app.

## Required: Add Input
- `paultimothymooney/chest-xray-pneumonia`

## Required: Secrets
- `HF_TOKEN` — HuggingFace token with MedGemma access
- `GROQ_API_KEY` — from console.groq.com (only needed for QA generation)
- `NGROK_TOKEN` — from dashboard.ngrok.com (only needed for public demo URL)

## Settings → Accelerator → GPU T4 x2

## Step 1 — Install Packages

In [ ]:
!pip install -q groq tqdm pandas
!pip install -q --upgrade peft transformers
!pip install -q colpali-engine accelerate bitsandbytes
!pip install -q open-clip-torch faiss-cpu
!pip install -q gradio pyngrok
!pip install -q --upgrade torchao
print('✓ All packages installed')

## Step 2 — Load Secrets & Clone Repo

In [ ]:
import os, sys, subprocess, glob, gc, time
import torch

os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

WORKING_DIR       = '/kaggle/working'
COLPALI_INDEX_DIR = os.path.join(WORKING_DIR, 'colpali_index')
CLIP_INDEX_DIR    = os.path.join(WORKING_DIR, 'clip_index')
os.makedirs(COLPALI_INDEX_DIR, exist_ok=True)
os.makedirs(CLIP_INDEX_DIR,    exist_ok=True)

# Load Kaggle secrets
from kaggle_secrets import UserSecretsClient
secrets      = UserSecretsClient()
HF_TOKEN     = secrets.get_secret('HF_TOKEN')
NGROK_TOKEN  = secrets.get_secret('NGROK_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN

try:
    GROQ_API_KEY = secrets.get_secret('GROQ_API_KEY')
except Exception:
    GROQ_API_KEY = ''
    print('⚠ GROQ_API_KEY not found — QA generation will be skipped')

# Clone repo
REPO_PATH = os.path.join(WORKING_DIR, 'DSAI413-A2')
if not os.path.exists(REPO_PATH):
    subprocess.run(
        ['git', 'clone', '-q', 'https://github.com/BASEL213/DSAI413-A2.git', REPO_PATH],
        check=True,
    )
else:
    subprocess.run(['git', '-C', REPO_PATH, 'pull', '-q'], check=True)

sys.path.insert(0, REPO_PATH)
print('✓ Secrets loaded')
print('✓ Repo ready')

## Step 3 — Detect Dataset

In [ ]:
# Find dataset root (contains train/val/test)
search_dirs = glob.glob('/kaggle/input/**/train/NORMAL', recursive=True)

if not search_dirs:
    raise RuntimeError(
        'Dataset not found!\n'
        'Add it via: + Add Input → paultimothymooney/chest-xray-pneumonia'
    )

DATASET_ROOT = os.path.dirname(os.path.dirname(search_dirs[0]))
print(f'✓ Dataset root: {DATASET_ROOT}')

for split in ('train', 'val', 'test'):
    split_dir = os.path.join(DATASET_ROOT, split)
    if os.path.isdir(split_dir):
        n_normal    = len(glob.glob(os.path.join(split_dir, 'NORMAL', '*')))
        n_pneumonia = len(glob.glob(os.path.join(split_dir, 'PNEUMONIA', '*')))
        print(f'  {split:5s}: {n_normal} NORMAL | {n_pneumonia} PNEUMONIA')

## Step 4 — Load Dataset & Build Corpus CSV

In [ ]:
import pandas as pd
from src.data.pneumonia_loader import PneumoniaLoader

loader = PneumoniaLoader(images_dir=DATASET_ROOT)
df     = loader.load()
train_df, val_df, test_df = loader.train_val_test()

CORPUS_PATH = os.path.join(WORKING_DIR, 'reports_corpus.csv')
df.to_csv(CORPUS_PATH, index=False)

print(f'Total : {len(df)}')
print(f'Train : {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
print(f'Saved : {CORPUS_PATH}')

## Step 5 — Generate QA Pairs via Groq (Optional)
Skip this cell if you don't have a GROQ_API_KEY.

In [ ]:
if not GROQ_API_KEY:
    print('⚠ Skipping QA generation — no GROQ_API_KEY')
else:
    from src.data.qa_creator import QACreator

    QA_PATH = os.path.join(WORKING_DIR, 'qa_dataset.jsonl')
    creator = QACreator(groq_api_key=GROQ_API_KEY)
    pairs   = creator.generate_dataset(df=train_df, output_path=QA_PATH, max_studies=200)
    print(f'✓ Generated {len(pairs)} QA pairs → {QA_PATH}')

## Step 6 — Build ColPali Index

In [ ]:
MAX_IMAGES = 1000  # ← increase for better retrieval quality

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
print(f'VRAM free: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB / {torch.cuda.mem_get_info()[1]/1e9:.2f} GB')

corpus_images = [p for p in df['image_path'].tolist() if os.path.exists(p)][:MAX_IMAGES]
print(f'Indexing {len(corpus_images)} images with ColPali...')

from src.retrieval.colpali_retriever import ColPaliRetriever

start     = time.time()
colpali   = ColPaliRetriever()
colpali.build_index(image_paths=corpus_images, index_save_dir=COLPALI_INDEX_DIR)
elapsed   = time.time() - start
print(f'✓ ColPali index built in {elapsed/60:.1f} min')

## Step 7 — Build CLIP Index

In [ ]:
# Free ColPali memory before loading CLIP
if 'colpali' in dir():
    del colpali
gc.collect()
torch.cuda.empty_cache()
print(f'VRAM free: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB')

from src.retrieval.clip_retriever import CLIPRetriever

print(f'Building CLIP index on {len(corpus_images)} images...')
clip = CLIPRetriever()
clip.build_index(corpus_images, batch_size=128)
clip.save_index(CLIP_INDEX_DIR)
print('✓ CLIP index built')

## Step 8 — Launch Demo App (Gradio + ngrok public URL)

In [ ]:
import importlib.util, threading

# Free CLIP before loading MedGemma
if 'clip' in dir():
    del clip
gc.collect()
torch.cuda.empty_cache()

# Unset INDEX_REPO so app_gradio skips the HF download
os.environ.pop('INDEX_REPO', None)

# Load Gradio app module
spec    = importlib.util.spec_from_file_location('app_gradio', os.path.join(REPO_PATH, 'app', 'app_gradio.py'))
app_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(app_mod)

# Inject corpus from local CSV
corpus_df = pd.read_csv(CORPUS_PATH)
app_mod._study_id_to_impression = dict(zip(corpus_df['study_id'].astype(str), corpus_df['impression']))
app_mod._path_to_impression     = dict(zip(corpus_df['image_path'].astype(str), corpus_df['impression']))

# Override retriever loaders to use local indexes
def _get_colpali_local():
    # ColPali needs ~6 GB VRAM; with MedGemma loaded there is insufficient room on the T4.
    raise RuntimeError(
        'ColPali cannot load alongside MedGemma on a T4 (insufficient VRAM). '
        'Please select CLIP mode.'
    )

def _get_clip_local():
    if app_mod._clip is None:
        from src.retrieval.clip_retriever import CLIPRetriever
        # Use CPU so CLIP does not consume VRAM needed by MedGemma inference
        app_mod._clip = CLIPRetriever(device='cpu')
        app_mod._clip.load_index(CLIP_INDEX_DIR)
    return app_mod._clip

def _get_generator_local():
    if app_mod._generator is None:
        from src.generation.medgemma_generator import MedGemmaGenerator
        app_mod._generator = MedGemmaGenerator(hf_token=HF_TOKEN, load_in_4bit=True)
    return app_mod._generator

app_mod.get_colpali   = _get_colpali_local
app_mod.get_clip      = _get_clip_local
app_mod.get_generator = _get_generator_local

# Pre-load CLIP (CPU) and MedGemma (GPU) so the first request is fast
print('Pre-loading CLIP retriever on CPU...')
_get_clip_local()
print('CLIP ready (CPU)')
print('Pre-loading MedGemma (~60s)...')
_get_generator_local()
print(f'MedGemma ready | VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB / {torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB')
print('Models loaded - app ready to launch')

In [ ]:
import gradio as gr

# Close any previous Gradio instance to free port 7860
try:
    gr.close_all()
    time.sleep(2)
except Exception:
    pass

# Launch Gradio + ngrok
def run_gradio():
    app_mod.demo.launch(server_port=7860, share=False, server_name='0.0.0.0', quiet=True)

threading.Thread(target=run_gradio, daemon=True).start()
print('Starting Gradio... (10 sec)')
time.sleep(10)

ngrok.kill()
ngrok.set_auth_token(NGROK_TOKEN)
public_url = ngrok.connect(7860)

print()
print('='*60)
print(f'PUBLIC URL: {public_url}')
print('='*60)
print('Models pre-loaded - requests respond in ~5s.')
print('NOTE: ColPali mode is disabled on T4 (insufficient VRAM). Use CLIP mode.')

In [ ]:
# Run this cell to stop the demo
ngrok.kill()
app_mod.demo.close()
print('✓ Demo stopped')